# 第一次实现小网络：从手算到 NumPy 训练

[返回本周路线](README.md) · [前期总览](../README.md) · [最小参考实现](numpy_mlp.py) · [下一本：PyTorch 对照与训练](02_pytorch_training_walkthrough.ipynb)

这本 Notebook 对应 **W3-01～W3-07**。第一次做时按节走：先预测结果，再运行示例，最后改一个地方。每节已把本次需要的概念、公式、带注释代码、预期现象和排错入口放在一起；不需要先通读一整门大学课程。

示例默认可从上到下直接运行，`TODO` 只在独立练习中出现，不会中断演示。**“示例全部运行通过”只证明环境和参考实现可用，不能代替你自己的推导、改写与解释。** 请在根目录 `work/intro-network/` 保留自己的 `network.ipynb` 和实验记录；如果复制本文件到那里，本地相对文档链接仍以这份原版为导航。

| 学习单元 | 今天只解决的问题 | 独立留下的证据 | 用到时再翻的本地材料 |
|---|---|---|---|
| [W3-01](#w3-01) | 一个 batch 如何经过线性层与 ReLU？ | shape 表、一个手算元素 | [线代 §1](../../../course/materials/01_linear_algebra.md)、[网络 §1](../../../course/materials/10_neural_networks_and_backprop.md) |
| [W3-02](#w3-02) | 链式法则怎样变成一次更新？ | 手算梯度、3 条学习率轨迹 | [微积分 §1～3](../../../course/materials/03_calculus_optimization.md) |
| [W3-03](#w3-03) | 如何保存前向的中间量？ | 自己改写的 forward、shape 检查 | [网络 §4](../../../course/materials/10_neural_networks_and_backprop.md) |
| [W3-04](#w3-04) | logits 如何变成稳定的 loss？ | 常数平移、均匀概率等检查 | [网络 §2](../../../course/materials/10_neural_networks_and_backprop.md) |
| [W3-05](#w3-05) | 梯度如何逐层传回来？ | 含 shape 的反向推导 | [网络 §3](../../../course/materials/10_neural_networks_and_backprop.md) |
| [W3-06](#w3-06) | 怎样知道反传写对了？ | 全参数梯度检查表 | [微积分 §5](../../../course/materials/03_calculus_optimization.md) |
| [W3-07](#w3-07) | 实现能否学会 32 个训练样本？ | 划分、预处理、曲线与解释 | [网络 §5](../../../course/materials/10_neural_networks_and_backprop.md) |

每单元暂按 90～120 分钟；一次可以跨天。每次开始先重跑前置代码，结束写下“我解释得了什么、哪里还依赖示例”。W3-07 到此只排查实现与优化；最终测试评价留给下一本的 W3-10。


## 0｜环境和文件放在哪里

选择项目原有 `.venv` 的 Python 内核，只需要 NumPy、scikit-learn、Matplotlib；本册不依赖 PyTorch，不下载数据。环境问题先看 [启动说明](../../../docs/setup/README.md)。

下面从当前工作目录向上寻找项目根目录，因此从项目根目录、本周目录或 `work/intro-network/` 打开都能运行。示例产物统一写入 `assignments/bootcamp/week3/outputs/numpy_tutorial/`，同名文件会在重跑时更新。自己的实验另存到 `work/intro-network/`，不要把演示产物当成已完成作业。


In [ ]:
from pathlib import Path
import csv
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "assignments/bootcamp/week3/numpy_mlp.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("请从这个项目内部启动 Notebook；详见根目录 README.md。")
WEEK3 = ROOT / "assignments/bootcamp/week3"
OUT = WEEK3 / "outputs/numpy_tutorial"
OUT.mkdir(parents=True, exist_ok=True)
if str(WEEK3) not in sys.path:
    sys.path.insert(0, str(WEEK3))
from numpy_mlp import (init_params, forward, softmax_cross_entropy,
                       loss_and_gradients, sgd_step, predict)

np.set_printoptions(precision=5, suppress=True)
SEED = 42
print("Python:", sys.executable)
print("示例输出:", OUT)


<a id="w3-01"></a>
## W3-01｜先手算一个输出，再检查整个 shape

每行是一个样本：`X` 的 shape 是 `(B, D)`。本例 `B=2` 个样本、每个 `D=3` 个输入、`H=4` 个隐藏单元、`K=2` 个输出类别。

$$Z_1=XW_1+b_1,\qquad H_1=\max(0,Z_1),\qquad S=H_1W_2+b_2.$$

`@` 是矩阵乘法，`*` 是逐元素乘法。矩阵乘法要对上中间维数 `(B,D) @ (D,H)`；`b1` 的 shape 是 `(H,)`，会在每一行上加同一组偏置，这就是这里的广播。

先看下一格中的整数。第 1 个样本、第 3 个隐藏单元（Python 下标 `[0,2]`）为：

$$1\times0+2\times2+(-1)\times(-1)+(-1)=4.$$

先不运行：预测整个 `Z1` 的 shape，并手算 `[1,3]`。ReLU 之后哪些负数会变成 0？


In [ ]:
X_shape = np.array([[1., 2., -1.], [0., -1., 2.]])   # (2, 3)
W_shape = np.array([[1., -1., 0., 2.],
                    [0., 1., 2., -1.],
                    [1., 0., -1., 1.]])             # (3, 4)
b_shape = np.array([0., 1., -1., 0.])                 # (4,)

Z_shape = X_shape @ W_shape + b_shape
H_shape = np.maximum(Z_shape, 0.0)
print("线性层输出:\n", Z_shape)
print("ReLU 输出:\n", H_shape)
assert Z_shape.shape == (2, 4)
np.testing.assert_allclose(Z_shape, [[0, 2, 4, -1], [2, 0, -5, 3]])
assert Z_shape[0, 2] == 4

W2_shape = np.array([[1., 0.], [0., 1.], [1., -1.], [-1., 1.]])
b2_shape = np.zeros(2)
S_shape = H_shape @ W2_shape + b2_shape
for name, value in {"X": X_shape, "W1": W_shape, "b1": b_shape,
                    "Z1": Z_shape, "H1": H_shape, "W2": W2_shape,
                    "b2": b2_shape, "logits": S_shape}.items():
    print(f"{name:>6}: {value.shape}")


**如何读结果：** `Z1[1,3]` 应为 3，输出 logits 是 `(2,2)`。logit 是模型给每类的原始分数，可以是负数，也不要求加起来等于 1。

隐藏层为什么需要激活？如果去掉 ReLU，$(XW_1+b_1)W_2+b_2=X(W_1W_2)+(b_1W_2+b_2)$，仍然是一次仿射变换。多堆一层没有得到这里想要的非线性表达能力。

**现在自己做：** 在自己的单元里，把 batch 改成 3 行，输入仍为 3 维。先写 shape 表，再运行；只改变一个偏置，预测它会影响哪一列。

<details><summary>卡住时看提示</summary>

3 行输入会得到 `(3,4)` 的隐藏值和 `(3,2)` 的 logits；参数 shape 不变。修改 `b1[2]` 会改变全部样本的第 3 个隐藏单元。若出现维度错误，逐个打印 shape，先检查矩阵乘法中间两个维数，而不是尝试随意 reshape。

</details>


In [ ]:
# TODO（独立练习，不影响后续演示）：在自己的 Notebook 重写一组 3 行输入。
# 先在纸上写 X、W1、b1、Z1、H1、W2、b2、logits 的 shape，再运行验证。
# 最后用两句话解释 @ 与 *，以及 b1 在哪一维广播。


<a id="w3-02"></a>
## W3-02｜把链式法则拆成两段

先不碰网络。令 $z=2w$、$L=(z-3)^2$，则

$$\frac{dL}{dz}=2(z-3),\qquad \frac{dz}{dw}=2,
\qquad \frac{dL}{dw}=4(2w-3).$$

从 `w=0` 出发：`z=0`，`L=9`，`dL/dw=-12`。学习率为 `0.1` 时，`w_new = 0 - 0.1 × (-12) = 1.2`，新损失是 `0.36`。

“算出梯度”和“按梯度更新”是两件事。梯度是局部变化率，不是损失本身；减去梯度给出了足够小步长时的下降方向。步子太大就可能越过低点并发散。


In [ ]:
w = 0.0
z = 2 * w
loss_before = (z - 3) ** 2
# 反着沿计算图走：先求 dL/dz，再乘 dz/dw。
dloss_dz = 2 * (z - 3)
dz_dw = 2.0
gradient = dloss_dz * dz_dw
w_after = w - 0.1 * gradient
loss_after = (2 * w_after - 3) ** 2
print(f"w: {w} → {w_after}; gradient: {gradient}; loss: {loss_before} → {loss_after}")
np.testing.assert_allclose([gradient, w_after, loss_after], [-12, 1.2, 0.36])


In [ ]:
trajectories = {}
for lr in [0.01, 0.1, 1.0]:
    w = 0.0
    losses = [(2 * w - 3) ** 2]  # 第 0 步：尚未更新
    weights = [w]
    for step in range(20):
        grad = 4 * (2 * w - 3)
        w = w - lr * grad
        weights.append(w)
        losses.append((2 * w - 3) ** 2)
    trajectories[lr] = {"weights": weights, "losses": losses}
    print(f"lr={lr}: final w={w:.6g}, final loss={losses[-1]:.6g}")

fig, ax = plt.subplots(figsize=(7, 3.5))
for lr, trajectory in trajectories.items():
    # 只为对数轴画图加一个显示下限，不改变实际优化或记录的 loss。
    ax.semilogy(np.maximum(trajectory["losses"], 1e-30), label=f"lr={lr}")
ax.set(xlabel="Update step", ylabel="Loss (log scale)", title="One parameter, three learning rates")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "scalar_learning_rates.png", dpi=150)
plt.show()


**预期现象：** `0.01` 较慢靠近最优点 `w=1.5`，`0.1` 更快靠近，`1.0` 发散。这里可以精确推导：$w_{t+1}-1.5=(1-8\eta)(w_t-1.5)$，所以本函数的收敛范围为 $0<\eta<0.25$；这个数值范围不能照搬到别的损失上。

**自己做：** 不看示例，推导 `L=(3w-2)²` 的梯度，并从 `w=0` 手算一步。任选两个学习率，先预测谁可能不稳定，再运行 20 步。保留失败轨迹，不只保留漂亮的曲线。

<details><summary>检查你的推导</summary>

梯度是 `6*(3*w-2)`。遗漏外层平方的系数 2，或遗漏内层 `3*w` 的导数 3，都会给出错误结果。此函数最优点是 `w=2/3`，收敛范围是 `0 < lr < 1/9`。请先推导再核对。

</details>


In [ ]:
# TODO：在自己的 Notebook 推导并实现 L=(3w-2)**2，记录手算和数值结果。
# 写一句话：为何“gradient 是负数”不等于“loss 是负数”？


<a id="w3-03"></a>
## W3-03｜把两层前向写成可复用函数

先沿用已经理解的三行运算，再谈封装。缓存 `X、z1、h1` 是为了反传：第一层求梯度需要输入 `X`；ReLU 求导需要知道激活前是正还是负；第二层求梯度需要隐藏输出 `h1`。

权重需要随机初始化以打破隐藏单元的对称性，偏置可以先全为 0。参考实现第一层使用 `sqrt(2/D)` 缩放，第二层使用 `sqrt(1/H)`；本次先理解“尺度随输入维数调整”，详细初始化理论留到后续阶段。固定种子使重新初始化可复现，不表示训练过程应该每一步都重置随机生成器。


In [ ]:
# 把刚刚手算过的运算逐行写出来，先不依赖 forward 函数。
rng = np.random.default_rng(SEED)
D, H, K = 3, 4, 2
params_demo = {
    "W1": rng.normal(size=(D, H)) * np.sqrt(2.0 / D),
    "b1": np.zeros(H),
    "W2": rng.normal(size=(H, K)) * np.sqrt(1.0 / H),
    "b2": np.zeros(K),
}
z1_demo = X_shape @ params_demo["W1"] + params_demo["b1"]
h1_demo = np.maximum(z1_demo, 0.0)
logits_demo = h1_demo @ params_demo["W2"] + params_demo["b2"]
cache_demo = {"X": X_shape, "z1": z1_demo, "h1": h1_demo}
print("logits:\n", logits_demo)
print("参数个数:", sum(value.size for value in params_demo.values()))
assert sum(value.size for value in params_demo.values()) == D * H + H + H * K + K

# 模块只是把上述相同运算收进函数，方便后面的训练和 PyTorch 对照复用。
params_from_function = init_params(D, H, K, seed=SEED)
logits_from_function, cache = forward(X_shape, params_from_function)
for name in params_demo:
    np.testing.assert_allclose(params_demo[name], params_from_function[name])
np.testing.assert_allclose(logits_demo, logits_from_function)
assert forward(X_shape[:1], params_demo)[0].shape == (1, K)
assert forward(np.vstack([X_shape, X_shape]), params_demo)[0].shape == (4, K)
print("固定输入/参数与 batch shape 检查通过。")


**预期现象：** `3→4→2` 网络一共有 `3×4+4+4×2+2=26` 个参数。batch 大小变了，参数量不变。`forward` 返回 logits 和 cache，不修改任何权重。

**自己做：** 在自己的单元写 `my_forward(X, params)`，不要直接调用参考 `forward`；与参考函数比较后，再用 `3→5→2` 的网络验证 shape。写下为什么 `h1` 和 `z1` 都值得保留。

<details><summary>常见错误与提示</summary>

如果对相同输入连续调用前向得到不同结果，检查是否把随机初始化写到了 `forward` 内部。若返回概率而后面又算一次 softmax，会使损失的含义改变；这里始终返回原始 logits。`z1 > 0` 就是稍后反向传播的 ReLU mask。

</details>


In [ ]:
# TODO：写自己的 my_forward，并用 np.testing.assert_allclose 与参考结果核对。
# 请先逐行解释缓存用途，再打开 numpy_mlp.py 阅读封装后的版本。


<a id="w3-04"></a>
## W3-04｜从 logits 到稳定的交叉熵

对第 $i$ 个样本、第 $k$ 类的分数 $s_{ik}$，softmax 把它变成概率：

$$p_{ik}=\frac{e^{s_{ik}}}{\sum_j e^{s_{ij}}},\quad
L=-\frac1B\sum_i\log p_{i,y_i}.$$

直接算 `exp(1000)` 会溢出；先把每行减去该行最大值 $m_i$。分子分母同时乘 $e^{-m_i}$，概率不变。记 `q = logits - max(logits)`：

$$\log p_{ik}=q_{ik}-\log\sum_j e^{q_{ij}}.$$

这就是稳定的 log-sum-exp 写法。**从 log 概率取目标类别的损失**，不要先把概率四舍五入、截断，再取 log。极端错误预测可能让某个概率在浮点数里下溢成 0，但上述 log 概率仍然有限。


In [ ]:
logits_example = np.array([[2., 1., -1.], [0., 0., 0.]])
y_example = np.array([0, 2])  # 标签不是 one-hot，而是正确类别的列号

shifted = logits_example - logits_example.max(axis=1, keepdims=True)
exp_shifted = np.exp(shifted)
normalizer = exp_shifted.sum(axis=1, keepdims=True)
probabilities_example = exp_shifted / normalizer
log_probabilities_example = shifted - np.log(normalizer)
# arange 指定每一行，y 指定该行的正确类别列；最后对 batch 求平均。
per_example_loss = -log_probabilities_example[np.arange(len(y_example)), y_example]
loss_example = per_example_loss.mean()
print("每行类别概率:\n", probabilities_example)
print("每样本损失:", per_example_loss, "平均损失:", loss_example)

reference_loss, reference_probabilities = softmax_cross_entropy(logits_example, y_example)
np.testing.assert_allclose(reference_probabilities, probabilities_example)
np.testing.assert_allclose(reference_loss, loss_example)


In [ ]:
# 这些是不依赖“模型学没学好”的数值性质，先通过再训练。
base_loss, base_probs = softmax_cross_entropy(logits_example, y_example)
shift_loss, shift_probs = softmax_cross_entropy(logits_example + 1000.0, y_example)
np.testing.assert_allclose(base_probs.sum(axis=1), 1.0)
np.testing.assert_allclose(base_probs, shift_probs, atol=1e-12)
np.testing.assert_allclose(base_loss, shift_loss, atol=1e-12)

uniform_loss, uniform_probs = softmax_cross_entropy(np.zeros((4, 3)), np.array([0, 1, 2, 0]))
np.testing.assert_allclose(uniform_probs, 1 / 3)
np.testing.assert_allclose(uniform_loss, np.log(3))

confident_correct, _ = softmax_cross_entropy(np.array([[10., -10., -10.]]), np.array([0]))
confident_wrong, _ = softmax_cross_entropy(np.array([[10., -10., -10.]]), np.array([1]))
extreme_loss, extreme_probs = softmax_cross_entropy(np.array([[1000., -1000.]]), np.array([1]))
assert confident_correct < confident_wrong
assert np.isfinite(extreme_loss) and np.isfinite(extreme_probs).all()
np.testing.assert_allclose(extreme_loss, 2000.0)

print("均匀三类损失:", uniform_loss, "≈ log(3)")
print("正确且自信:", confident_correct, "; 错误且自信:", confident_wrong)
print("极端错误预测 loss:", extreme_loss, "; probabilities:", extreme_probs)
print("概率和、平移不变性、log(K) 与有限值检查通过。")


**自己做：** 改为 5 类，把所有 logits 设为 7，先预测 loss 再运行；接着给每行分别加不同的常数，检查结果是否仍相同。写下为什么“错误且自信”的损失更大。

<details><summary>提示和排错</summary>

均匀 5 类时为 `log(5)`。每一行可以减去自己的常数，不需要全 batch 同一个常数。若概率沿列求和不为 1，检查 `axis=1, keepdims=True`；若索引报错，检查标签是一维整数且在 `0..K-1`；若极端 logits 出现 Inf，检查是否仍在直接 `exp(logits)`。

</details>


In [ ]:
# TODO：补上 5 类、每行不同常数的检查；先写预期，再写代码。
# 注意：全相等 logits 的 loss 才恰好是 log(K)；任意随机初始化不保证完全相等。


<a id="w3-05"></a>
## W3-05｜沿计算图倒着写梯度

先看最末端。对单个样本，$L_i=-s_{i,y_i}+\log\sum_k e^{s_{ik}}$；对某个 logit 求导，第一项给出“正确类减 1”，第二项给出 softmax 概率。batch 平均再除以 $B$：

$$dS=\frac{P-Y}{B}.$$

`Y` 是 one-hot 的数学表示；代码可以用“在正确类别上减 1”省去显式构造。然后把每层输入的影响乘回来：

| 反向的量 | 公式/代码 | shape |
|---|---|---|
| `dlogits` | `(P - Y) / B` | `(B,K)` |
| `dW2` | `h1.T @ dlogits` | `(H,K)` |
| `db2` | `dlogits.sum(axis=0)` | `(K,)` |
| `dh1` | `dlogits @ W2.T` | `(B,H)` |
| `dz1` | `dh1 * (z1 > 0)` | `(B,H)` |
| `dW1` | `X.T @ dz1` | `(D,H)` |
| `db1` | `dz1.sum(axis=0)` | `(H,)` |

为什么 bias 要沿 batch 求和？同一个 `b[k]` 被加到了每个样本上，每条路径都对损失有贡献，要把它们加起来。平均系数已包含在 `dlogits` 中，后面不再除一次。


In [ ]:
y_demo = np.array([0, 1])
loss_demo, probs_demo = softmax_cross_entropy(logits_demo, y_demo)

# 最末端：softmax + 平均 cross-entropy 的合成导数。
dlogits = probs_demo.copy()             # copy 避免把概率原地改坏
dlogits[np.arange(len(y_demo)), y_demo] -= 1.0
dlogits /= len(y_demo)                  # 整条反传只在这里平均一次

# 第二层，再穿过 ReLU，最后到第一层；按依赖顺序逐行查看 shape。
dW2_demo = h1_demo.T @ dlogits
db2_demo = dlogits.sum(axis=0)
dh1_demo = dlogits @ params_demo["W2"].T
dz1_demo = dh1_demo * (z1_demo > 0.0)
dW1_demo = X_shape.T @ dz1_demo
db1_demo = dz1_demo.sum(axis=0)
grads_demo = {"W1": dW1_demo, "b1": db1_demo, "W2": dW2_demo, "b2": db2_demo}

loss_function, grads_function = loss_and_gradients(X_shape, y_demo, params_demo)
np.testing.assert_allclose(loss_function, loss_demo)
for name in params_demo:
    print(name, "参数:", params_demo[name].shape, "梯度:", grads_demo[name].shape)
    assert grads_demo[name].shape == params_demo[name].shape
    np.testing.assert_allclose(grads_demo[name], grads_function[name])


In [ ]:
# 反传完成后才更新。先 copy，保留原始参数供下一节解释/比较。
params_one_step = {name: value.copy() for name, value in params_demo.items()}
lr = 0.01
for name in params_one_step:
    params_one_step[name] -= lr * grads_demo[name]
loss_after_step, _ = loss_and_gradients(X_shape, y_demo, params_one_step)
print(f"一次更新前 loss={loss_demo:.6f}；更新后 loss={loss_after_step:.6f}")
assert loss_after_step < loss_demo  # 这是本例的 sanity check，不是反传正确性的证明。

# 核对封装后的更新函数执行的是完全相同的事情。
params_one_step_ref = {name: value.copy() for name, value in params_demo.items()}
sgd_step(params_one_step_ref, grads_demo, lr)
for name in params_one_step:
    np.testing.assert_allclose(params_one_step[name], params_one_step_ref[name])


**自己做：** 盖住代码，按 `loss → logits → h1 → z1 → 参数` 顺序重写梯度，给每行标 shape。解释 `dW2` 为什么要用 `h1.T`，以及为什么不能在算完 `dW2` 后立刻更新 `W2` 再算 `dh1`。

<details><summary>提示：两种常见错法</summary>

用更新后的 `W2` 来算 `dh1`，混用了前向时刻与更新后时刻的参数。重复除 batch 会让梯度过小。把 batch 完整复制一遍，平均 loss 和全部参数梯度都应保持不变，下面用这个性质核对。

</details>


In [ ]:
# 平均损失的性质：同一批数据重复两次，loss 与 gradient 都不应变化。
double_loss, double_grads = loss_and_gradients(
    np.vstack([X_shape, X_shape]), np.tile(y_demo, 2), params_demo)
np.testing.assert_allclose(double_loss, loss_demo)
for name in grads_demo:
    np.testing.assert_allclose(double_grads[name], grads_demo[name], atol=1e-12)
print("重复 batch 的平均 loss/gradient 不变。")
# TODO：独立重写反传，再用这个性质和下一节有限差分共同核对。


<a id="w3-06"></a>
## W3-06｜对全部 17 个参数做有限差分

损失下降一次仍可能藏着错误。现在换一个 `2→3→2` 的微型网络：`2×3+3+3×2+2=17` 个参数，逐个扰动，计算

$$g_j^{num}=\frac{L(\theta+\epsilon e_j)-L(\theta-\epsilon e_j)}{2\epsilon},\quad \epsilon=10^{-5}.$$

这不使用你的反传公式，因而可以独立核对。使用 float64，固定输入和参数，关闭所有随机行为。ReLU 在 0 处不可导；以下例子让激活前值离 0 有明确距离，并且对每次正负扰动都检查 mask 没变化。

按每个参数张量汇总：`relative_l2 = ||analytic - numeric|| / (||analytic|| + ||numeric|| + 1e-12)`，本例要求 `<1e-5`；同时查看 `max_abs_error`，因为近零梯度的相对误差不稳定。


In [ ]:
X_check = np.array([[0.2, -0.4], [0.7, 0.1], [-0.3, 0.8]], dtype=np.float64)
y_check = np.array([0, 1, 0])
params_check = {
    "W1": np.array([[0.4, -0.7, 0.3], [-0.2, 0.5, 0.6]], dtype=np.float64),
    "b1": np.array([0.5, -0.4, 0.2], dtype=np.float64),
    "W2": np.array([[0.3, -0.2], [0.6, 0.1], [-0.4, 0.5]], dtype=np.float64),
    "b2": np.array([0.1, -0.1], dtype=np.float64),
}
_, cache_check = forward(X_check, params_check)
base_mask = cache_check["z1"] > 0
print("检查点 z1:\n", cache_check["z1"])
print("离 ReLU 的 0 点最近距离:", np.abs(cache_check["z1"]).min())
assert np.abs(cache_check["z1"]).min() > 1e-3
assert sum(value.size for value in params_check.values()) == 17
check_loss, analytic_grads = loss_and_gradients(X_check, y_check, params_check)


In [ ]:
epsilon = 1e-5
gradient_check_rows = []
numeric_grads = {}
for name, tensor in params_check.items():
    numeric = np.zeros_like(tensor)
    # ndindex 遍历张量的全部下标，而不是只抽查几个元素。
    for index in np.ndindex(tensor.shape):
        original = tensor[index]
        try:
            tensor[index] = original + epsilon
            logits_plus, cache_plus = forward(X_check, params_check)
            loss_plus, _ = softmax_cross_entropy(logits_plus, y_check)
            np.testing.assert_array_equal(cache_plus["z1"] > 0, base_mask)

            tensor[index] = original - epsilon
            logits_minus, cache_minus = forward(X_check, params_check)
            loss_minus, _ = softmax_cross_entropy(logits_minus, y_check)
            np.testing.assert_array_equal(cache_minus["z1"] > 0, base_mask)
        finally:
            # 无论检查是否失败，都恢复原值，防止影响后续参数的检查。
            tensor[index] = original
        numeric[index] = (loss_plus - loss_minus) / (2 * epsilon)

    analytic = analytic_grads[name]
    absolute_error = float(np.abs(analytic - numeric).max())
    relative_l2 = float(np.linalg.norm(analytic - numeric) /
                        (np.linalg.norm(analytic) + np.linalg.norm(numeric) + 1e-12))
    numeric_grads[name] = numeric
    row = {"parameter": name, "elements": tensor.size,
           "relative_l2": relative_l2, "max_abs_error": absolute_error}
    gradient_check_rows.append(row)
    print(f"{name}: relative L2={relative_l2:.3e}, max abs={absolute_error:.3e}")
    assert relative_l2 < 1e-5
    assert absolute_error < 1e-7
    near_zero = np.abs(analytic) < 1e-8
    assert np.all(np.abs(analytic[near_zero] - numeric[near_zero]) < 1e-7)

with (OUT / "gradient_check.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=gradient_check_rows[0].keys())
    writer.writeheader()
    writer.writerows(gradient_check_rows)
print("全部 17 个参数通过；已保存 gradient_check.csv。")


**自己做：** 在自己的反传副本里故意把 `dlogits /= batch_size` 去掉，观察哪个张量首先失败、误差大约有什么规律，然后修好。不要改公共 `numpy_mlp.py`，以免下一本的对照也一起被改坏。

<details><summary>失败时按这个顺序定位</summary>

先看 loss 是否为 batch 平均、平均系数是否只用一次；再看 `dW` 的转置、`db` 的求和轴、ReLU 是否使用前向缓存。确认未跨 ReLU 0 点、参数已恢复、数组为 float64。遗漏 `/B` 时，解析梯度会变成数值梯度的 B 倍。不要通过不断增大容差让检查“通过”。

</details>

**本节出口：** 保存检查表，并能解释有限差分为什么不依赖解析反传。只有找到并解决差异后，才进入真实数据训练。


In [ ]:
# TODO：在独立副本中引入一个反传错误，记录失败表；修复后重新通过检查。
# 写清楚：这个检查验证了什么？为什么仍不能证明数据划分/任务定义正确？


<a id="w3-07"></a>
## W3-07｜先学会 32 个样本，再谈泛化

这次使用 `sklearn.datasets.load_digits()`：每个样本是一张 `8×8` 的灰度数字图像，展平后为 64 个输入，目标类别为 `0..9`。数据随 scikit-learn 提供，无需联网。

先划分原始样本的**索引**，再用训练集拟合标准化。固定种子 42，两次分层拆分得到约 `60%/20%/20%` 的 train/validation/test。分层只为让类别比例相近，不能替代对数据来源或泄漏的检查。本例是教学数据；现实任务如果同一用户有多条记录，通常还需按用户或时间拆分。

本节只取训练集里的 32 个样本做排错。验证集和测试集索引先保存，**不参与这次小样本训练，也不计算它们的表现**。训练准确率接近 100% 只能说明这组样本能被拟合，不能说明模型在新样本上有效。


In [ ]:
digits = load_digits()
X_raw = digits.data.astype(np.float64)   # (1797, 64)，像素值保留原始尺度
y_all = digits.target.astype(np.int64)
all_idx = np.arange(len(y_all))
train_idx, heldout_idx = train_test_split(
    all_idx, test_size=0.4, random_state=SEED, stratify=y_all)
val_idx, test_idx = train_test_split(
    heldout_idx, test_size=0.5, random_state=SEED, stratify=y_all[heldout_idx])

assert len(np.intersect1d(train_idx, val_idx)) == 0
assert len(np.intersect1d(train_idx, test_idx)) == 0
assert len(np.intersect1d(val_idx, test_idx)) == 0
np.testing.assert_array_equal(np.sort(np.concatenate([train_idx, val_idx, test_idx])), all_idx)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_raw[train_idx])  # 只在训练数据上学习 mean/scale
# 验证/测试集以后只能使用这同一个 scaler.transform；不能再 fit。
y_train = y_all[train_idx]
np.testing.assert_allclose(scaler.mean_, X_raw[train_idx].mean(axis=0))
print("train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
print("训练集标签计数:", np.bincount(y_train, minlength=10))
print("训练特征 shape:", X_train.shape)


In [ ]:
# 索引和预处理参数是下一本复现实验的依据；这里不保存测试评价结果。
np.savez(OUT / "digits_split_and_scaler.npz",
         train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
         mean=scaler.mean_, scale=scaler.scale_)

# 从训练集固定抽 32 个样本。这里用分层抽样，确保十个类别都有代表。
tiny_positions, _ = train_test_split(
    np.arange(len(y_train)), train_size=32, random_state=SEED, stratify=y_train)
tiny_idx = train_idx[tiny_positions]
X_tiny = X_train[tiny_positions]
y_tiny = y_train[tiny_positions]
np.testing.assert_array_equal(np.unique(y_tiny), np.arange(10))
np.save(OUT / "tiny_train_indices.npy", tiny_idx)

fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for ax, original_index in zip(axes.flat, tiny_idx[:10]):
    # 用原始图像展示：标准化后的数值适合训练，不直接当灰度显示。
    ax.imshow(digits.images[original_index], cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"label={y_all[original_index]}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "tiny_train_examples.png", dpi=150)
plt.show()
print("小样本 shape:", X_tiny.shape, "; 类别计数:", np.bincount(y_tiny, minlength=10))


### 训练循环只做四件事

1. 用当前参数前向并计算 loss；
2. 计算全部梯度；
3. 用同一组梯度更新全部参数；
4. 记录更新后的 loss 和 accuracy。

这里的网络是 `64→32→10`，使用前面已逐行解释过的参考函数，全 batch 梯度下降，学习率 `0.1`，500 次更新，不加正则化。函数名 `sgd_step` 表示同一个梯度更新公式；本例每步使用全部 32 个样本，所以具体是全 batch GD。第 0 行指标是未更新状态，第 t 行是完成 t 次更新后的状态，避免把两个时间点的 loss/accuracy 混在一起。


In [ ]:
config = {"example_only": True, "seed": SEED, "input_dim": 64, "hidden_dim": 32,
          "output_dim": 10, "tiny_train_size": 32, "learning_rate": 0.1,
          "updates": 500, "dtype": "float64", "optimizer": "full_batch_GD",
          "split": "stratified index split, 60/20/20 approximately, seed 42",
          "scaler_fit": "training partition only", "test_evaluated": False}
params_tiny = init_params(64, 32, 10, seed=SEED)
history = []
for step in range(config["updates"] + 1):
    # 每一行先记录当前参数的指标；step=0 还没有做任何更新。
    loss, gradients = loss_and_gradients(X_tiny, y_tiny, params_tiny)
    accuracy = float((predict(X_tiny, params_tiny) == y_tiny).mean())
    history.append({"step": step, "train_loss": loss, "train_accuracy": accuracy})
    assert np.isfinite(loss)
    assert all(np.isfinite(g).all() for g in gradients.values())
    if step % 100 == 0:
        print(f"step={step:3d}, tiny train loss={loss:.5f}, accuracy={accuracy:.3f}")
    if step < config["updates"]:
        sgd_step(params_tiny, gradients, config["learning_rate"])

# 只对这个固定的教学例子检查预期，不代表对任意数据/学习率的承诺。
assert history[-1]["train_loss"] < history[0]["train_loss"]
assert history[-1]["train_accuracy"] >= 0.97
print("固定教学小样本已接近完全拟合；这不是验证或测试准确率。")


In [ ]:
steps = [row["step"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(steps, [row["train_loss"] for row in history])
axes[0].set(xlabel="Update step", ylabel="Mean cross-entropy", title="Tiny train loss")
axes[1].plot(steps, [row["train_accuracy"] for row in history])
axes[1].set(xlabel="Update step", ylabel="Accuracy", ylim=(0, 1.02), title="Tiny train accuracy")
fig.tight_layout()
fig.savefig(OUT / "tiny_train_curves.png", dpi=150)
plt.show()

with (OUT / "tiny_train_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)
(OUT / "tiny_train_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
np.savez(OUT / "tiny_train_params.npz", **params_tiny)

# 重新加载参数，核对同一组训练样本预测未改变。
with np.load(OUT / "tiny_train_params.npz") as saved:
    loaded_params = {name: saved[name].copy() for name in params_tiny}
np.testing.assert_allclose(forward(X_tiny, loaded_params)[0], forward(X_tiny, params_tiny)[0])
print("保存产物:")
for path in sorted(OUT.iterdir()):
    if path.is_file():
        print("-", path.name)


### 先解释，再独立改写

**预期现象：** 固定例子的 loss 明显下降，32 个训练样本接近 100% 准确率。不要把曲线标题改成 validation 或 test：训练这些样本本来就是为了记住它们。

**你来改一处：** 在自己的 Notebook 中，将学习率改为 `0.01`，从相同初始参数重新训练 500 步，比较到达高训练准确率的速度。只改变学习率；不要沿用已经训练好的 `params_tiny`，也不要同时改隐藏层宽度。另存曲线、配置和你的解释，至少指出一次“相同训练表现不代表相同泛化表现”。

<details><summary>如果小样本仍学不会</summary>

依次核对：标签是否与输入索引同步，类别是否为 0～9；输入/参数/梯度 shape；W3-06 是否通过；loss/gradient 是否有限；学习率是否太大或太小；ReLU 是否大量一直为 0。先用这个已知可运行的配置定位差异，再修改容量。不能靠查看测试集或反复换测试划分来排错。

</details>


In [ ]:
# TODO：独立保存一个 lr=0.01、其余设置相同的实验，不覆盖演示输出。
# 使用 init_params(..., seed=42) 重新开始，而不是继续更新 params_tiny。
# 把“运行现象 → 我的解释 → 下一步验证”写成三句话。


## 本册完成后，带着什么进入下一册

参考演示已保存：`gradient_check.csv`、学习率示例图、数据划分与 scaler 参数、32 个训练样本索引、小样本曲线/历史/配置/参数。它们放在本周 `outputs/numpy_tutorial/`，重跑会更新；源码 Notebook 保持无执行输出，便于从新内核学习。

**自己的学习记录至少包含：** 一张 shape 表、一段链式法则推导、逐层反传说明、自己的梯度检查和一项单因素实验。用自己的话解释“数值梯度对照通过”与“模型泛化良好”之间还缺什么证据。

下一步打开 [W3-08～W3-10：PyTorch 对照与训练](02_pytorch_training_walkthrough.ipynb)，先让框架计算与这个 NumPy 实现对齐，再比较验证集表现、冻结选择、做最终评价。不要在本册追加最终测试分数。完整作业 [A06](../../core/A06/A06_backprop_and_training.md) 的多优化器等要求在后续阶段完成，本次不叠加。
